# Segmentación semántica de cultivos sobre PASTIS-R

Se entrenan dos modelos de segmentación densa sobre las series Sentinel-2 de PASTIS-R y se compara su desempeño píxel a píxel. El primero es una U-Net con encoder ResNet-50 sobre un composite temporal; el segundo toma AnySat como extractor congelado y entrena solo una cabeza lineal. La intención es ver cuánto rinde cada enfoque para asignar el tipo de cultivo a cada píxel de la parcela y quedarnos con el que mejor resultado dé.

El cuaderno está pensado para correr en Colab: monta el dataset desde Drive, lo copia al disco local de la sesión para que las épocas lean rápido, entrena los dos modelos y deja una tabla con las métricas y los tiempos de cada uno.

## Datos y métricas

PASTIS-R entrega parches Sentinel-2 multitemporales de 128x128, que aquí se reescalan a 256. Las etiquetas tienen 20 clases: fondo, 18 tipos de cultivo y una clase void que se descarta tanto en la pérdida como en las métricas. El split de entrenamiento y validación usa los folds oficiales del dataset, que son espacialmente disjuntos, de modo que parcelas vecinas no queden a la vez en entrenamiento y validación. Para comparar los modelos se reportan tres métricas a nivel de píxel: mIoU, F1-macro y exactitud.

In [1]:
# Setup del entorno. En Colab se monta Drive (donde vive el dataset) y se
# instalan las dependencias que no vienen por defecto; en local no hace falta.
import os, sys, subprocess
from pathlib import Path

_IN_COLAB = False
shared_folder_path = ''
try:
    from google.colab import drive
    drive.mount('/content/drive')
    shared_folder_path = '/content/drive/MyDrive/Integrador/'
    _IN_COLAB = True
except ImportError:
    pass

# En Colab el repo no esta presente: se clona una vez en /content/agrosat-copilot.
# Ajusta _branch si tu codigo esta en otra rama.
if _IN_COLAB:
    from getpass import getpass
    _repo_dir = '/content/agrosat-copilot'
    _branch = 'users/abocanegra/unet-anysat'
    _repo = 'github.com/ArthurZizumbo/agrosat-copilot.git'
    if not Path(_repo_dir, 'pyproject.toml').is_file():
        _rc = os.system(f'git clone --branch {_branch} --depth 1 https://{_repo} {_repo_dir}')
        if _rc != 0:  # repo privado: pide token (no se guarda en el notebook)
            _tok = getpass('GitHub token (repo privado): ')
            os.system(f'git clone --branch {_branch} --depth 1 https://{_tok}@{_repo} {_repo_dir}')

# El codigo no vive en Drive: se localiza el repo por su pyproject.toml.
_search = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
if _IN_COLAB:
    _search = [Path('/content/agrosat-copilot'), *_search]
for _cand in _search:
    if (_cand / 'pyproject.toml').is_file():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        os.chdir(_cand)
        break
else:
    raise RuntimeError('No se encontro el repo agrosat-copilot (pyproject.toml). '
                       'Clonalo en /content/agrosat-copilot o sincronizalo desde VS Code.')

if _IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'segmentation-models-pytorch', 'structlog', 'typer', 'polars', 'mlflow'], check=False)

print('repo:', Path.cwd(), '| colab:', _IN_COLAB, '| drive:', shared_folder_path or '(local)')

Mounted at /content/drive
repo: /content/agrosat-copilot | colab: True | drive: /content/drive/MyDrive/Integrador/


In [2]:
# Configuracion de la corrida.
import torch

# El dataset vive en Drive; en local se usa la copia del repo.
PASTIS_ROOT = Path((shared_folder_path + 'data/PASTIS-R') if shared_folder_path
                   else 'data/PASTIS-R')
# Las metricas de cada modelo se guardan en este parquet para armar la comparativa.
COMPARISON_PATH = Path((shared_folder_path if shared_folder_path else '')
                       + 'reports/segmentation/model_comparison_avance4_segmentacion.parquet')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TARGET_SIZE = 256
SUBSET = 0            # 0 = todos; reducir (p.ej. 60) si la sesion es corta
EPOCHS_UNET = 30
EPOCHS_ANYSAT = 30
MLFLOW_URI = 'file:./mlruns'
# Por defecto se lee directo de Drive (sin copiar). Varios workers leen en
# paralelo para compensar la latencia. Si vas a entrenar muchas epocas y
# preferis acelerar, pone COPY_TO_LOCAL=True (copia una vez al disco efimero).
COPY_TO_LOCAL = False
# Tamanos pensados para una GPU L4 (24 GB). En T4 (16 GB) bajar a la mitad.
BATCH_UNET = 16
BATCH_ANYSAT = 8
NUM_WORKERS = 4 if _IN_COLAB else 0

print('PASTIS_ROOT:', PASTIS_ROOT, '| exists:', PASTIS_ROOT.exists())
print('device:', DEVICE, '| target_size:', TARGET_SIZE, '| subset:', SUBSET)

PASTIS_ROOT: /content/drive/MyDrive/Integrador/data/PASTIS-R | exists: True
device: cuda | target_size: 256 | subset: 0


## Lectura del dataset

Por defecto el dataset se lee directo desde Drive, sin copiar nada: así se evita la espera inicial y no se pierde trabajo si la sesión se reinicia. Para que la lectura no sea un cuello de botella, el loader abre cada parche con un solo acceso a disco (no vuelve a leer el archivo de metadatos en cada paso) y el DataLoader usa varios procesos en paralelo.

Si vas a entrenar muchas épocas y preferís acelerar la lectura, poné `COPY_TO_LOCAL = True` en la celda anterior: copia una sola vez al disco local de la sesión y a partir de ahí lee desde SSD. Ese disco es efímero, así que si la sesión se reinicia hay que volver a copiar. En local esta celda no hace nada.

In [3]:
# Copia del dataset de Drive al disco local, con barra de progreso.
import shutil, time

def copy_pastis_to_local(src_root, dst_root,
                         subdirs=('DATA_S2', 'ANNOTATIONS'),
                         files=('metadata.geojson', 'NORM_S2_patch.json')):
    src_root, dst_root = Path(src_root), Path(dst_root)
    dst_root.mkdir(parents=True, exist_ok=True)
    todo = []
    for sub in subdirs:
        for f in sorted((src_root / sub).glob('*')):
            if f.is_file():
                todo.append((f, dst_root / sub / f.name))
    for fname in files:
        sp = src_root / fname
        if sp.is_file():
            todo.append((sp, dst_root / fname))
    if not todo:
        raise FileNotFoundError(f'No se hallaron DATA_S2/ANNOTATIONS en {src_root}')
    total_bytes = sum(s.stat().st_size for s, _ in todo)
    try:
        from tqdm.auto import tqdm
        bar = tqdm(total=total_bytes, unit='B', unit_scale=True, desc='Copiando PASTIS')
    except Exception:
        bar = None
    t0 = time.time()
    for i, (src, dst) in enumerate(todo, 1):
        dst.parent.mkdir(parents=True, exist_ok=True)
        # Salta el archivo si ya esta copiado con el mismo tamano.
        if not (dst.exists() and dst.stat().st_size == src.stat().st_size):
            shutil.copy2(src, dst)
        if bar is not None:
            bar.update(src.stat().st_size)
        elif i % 200 == 0:
            print(f'  {i}/{len(todo)} archivos...')
    if bar is not None:
        bar.close()
    print(f'Listo: {len(todo)} archivos ({total_bytes / 1e9:.1f} GB) en {time.time() - t0:.0f}s -> {dst_root}')
    return dst_root

if _IN_COLAB and COPY_TO_LOCAL:
    PASTIS_ROOT = copy_pastis_to_local(PASTIS_ROOT, '/content/PASTIS-R')
    print('PASTIS_ROOT (local):', PASTIS_ROOT, '| exists:', PASTIS_ROOT.exists())
else:
    print('Lectura directa desde:', PASTIS_ROOT, '| exists:', PASTIS_ROOT.exists())

Lectura directa desde: /content/drive/MyDrive/Integrador/data/PASTIS-R | exists: True


In [4]:
# Split en los folds oficiales de PASTIS (espacialmente disjuntos).
from ml.ingest.pastis_dataset import pastis_fold_split

split = pastis_fold_split(PASTIS_ROOT, train_folds=(1, 2, 3), val_folds=(4,), test_folds=(5,))
print({k: len(v) for k, v in split.items()})

{'train': 1455, 'val': 482, 'test': 496}


## U-Net con encoder ResNet-50

La primera arquitectura es una U-Net clásica. El encoder ResNet-50 viene preentrenado en ImageNet y se adapta a las diez bandas de Sentinel-2. Como entrada se usa la mediana temporal de la serie y la salida es un mapa de clases a la resolución de la imagen.

In [5]:
# Entrenamiento de la U-Net.
from ml.train.train_segmentation import run_training

unet_result = run_training(
    model='unet', epochs=EPOCHS_UNET, batch_size=BATCH_UNET, target_size=TARGET_SIZE,
    subset=SUBSET, device=DEVICE, root=PASTIS_ROOT, mlflow_uri=MLFLOW_URI,
    comparison_path=COMPARISON_PATH, num_workers=NUM_WORKERS,
)
unet_result

ModuleNotFoundError: No module named 'spyndex'

## AnySat congelado con cabeza lineal

La segunda arquitectura parte de AnySat (Astruc et al., 2024), un modelo fundacional para datos de observación de la Tierra. Aquí se usa congelado, como extractor de características, y solo se entrena una cabeza lineal que las proyecta a las clases de cultivo. El entrenamiento resulta mucho más barato porque el grueso de los pesos no se actualiza. AnySat se descarga la primera vez desde su repositorio; si esa descarga falla, la celda lo avisa sin detener el resto del cuaderno.

In [ ]:
# Carga de AnySat y entrenamiento de la cabeza lineal.
anysat_result = None
try:
    from ml.models.anysat_wrapper import load_anysat_encoder
    _ = load_anysat_encoder()  # descarga y valida los pesos antes de entrenar
    anysat_result = run_training(
        model='anysat', epochs=EPOCHS_ANYSAT, batch_size=BATCH_ANYSAT, target_size=TARGET_SIZE,
        subset=SUBSET, device=DEVICE, root=PASTIS_ROOT, mlflow_uri=MLFLOW_URI,
        comparison_path=COMPARISON_PATH, num_workers=NUM_WORKERS,
    )
except Exception as exc:
    print('AnySat no disponible en esta corrida:', exc)
    print('Revisa el acceso a torch.hub gastruc/anysat y reejecuta esta celda.')
anysat_result

## Comparativa

La tabla reúne las métricas de los dos modelos sobre el fold de validación, ordenadas por mIoU, junto con el tiempo de entrenamiento de cada uno.

In [ ]:
# --- Tabla comparativa ---
import polars as pl

table = pl.read_parquet(COMPARISON_PATH).sort('miou', descending=True)
cols = ['model', 'miou', 'f1_macro', 'pixel_accuracy', 'train_time_s', 'epochs', 'n_train', 'n_val']
table.select([c for c in cols if c in table.columns])

## Matriz de confusión

Recall por clase a nivel de píxel sobre el fold de validación, sin contar la clase void. Ayuda a ver qué cultivos se separan bien y cuáles se confunden entre sí.

In [ ]:
# Matriz de confusion a nivel de pixel.
import torch
from torch.utils.data import DataLoader
from ml.ingest.pastis_dataset import PASTISDataset, load_norm_stats, PASTIS_IGNORE_INDEX
from ml.ingest.pastis_loader import PASTIS_CLASS_MAP
from ml.eval.dense_metrics import dense_confusion_figure
from ml.models.segmentation import build_unet

def confusion_figure(model_name, reduction, build_fn, ckpt, max_patches=40):
    norm = load_norm_stats(PASTIS_ROOT, folds=(1, 2, 3))
    val_ids = split['val'][:max_patches]
    ds = PASTISDataset(val_ids, root=PASTIS_ROOT, target_size=TARGET_SIZE,
                       temporal_reduction=reduction, norm=norm)
    loader = DataLoader(ds, batch_size=2)
    model = build_fn().to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()
    preds, tgts = [], []
    with torch.no_grad():
        for b in loader:
            img = b['image'].to(DEVICE)
            out = model(img) if model_name == 'unet' else model(img, b['dates'].to(DEVICE))
            preds.append(out.argmax(1).cpu().reshape(-1))
            tgts.append(b['semantic'].reshape(-1))
    return dense_confusion_figure(torch.cat(preds), torch.cat(tgts),
                                  class_names=PASTIS_CLASS_MAP, ignore_index=PASTIS_IGNORE_INDEX)

fig_unet = confusion_figure('unet', 'median', lambda: build_unet(20, encoder_weights=None),
                            unet_result['checkpoint_path'])
fig_unet

## Conclusiones

La comparativa deja ver el contraste entre los dos enfoques. La U-Net trabaja sobre un resumen temporal de la serie y es la opción más directa; AnySat reutiliza un modelo ya entrenado y apenas ajusta una cabeza, con un costo de entrenamiento bastante menor. La matriz de confusión muestra dónde se concentra el error, que suele estar entre cultivos de la misma familia. Con estos resultados se elige el modelo que mejor desempeño dé y, si vale la pena, se afinan sus hiperparámetros con una búsqueda más fina como la del bloque siguiente.

In [ ]:
# Busqueda de hiperparametros con Optuna para el modelo elegido (opcional).
# Descomentar para ajustar el que haya dado mejor resultado.
#
# import optuna
# def objective(trial):
#     lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
#     wd = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
#     bs = trial.suggest_categorical('batch_size', [4, 8])
#     res = run_training(model='unet', epochs=15, batch_size=bs, lr=lr, weight_decay=wd,
#                        target_size=TARGET_SIZE, subset=SUBSET, device=DEVICE,
#                        root=PASTIS_ROOT, mlflow_uri=MLFLOW_URI)
#     return res['miou']
# study = optuna.create_study(direction='maximize', study_name='tune-unet')
# study.optimize(objective, n_trials=30)
# study.best_params